In [ ]:
from fastf1 import get_event
import fastf1
import pandas as pd
from concurrent.futures import ThreadPoolExecutor
import os
import logging
logging.getLogger('fastf1').setLevel(logging.ERROR)
os.makedirs('cache_folder', exist_ok=True)
fastf1.Cache.enable_cache('cache_folder')

In [2]:
aus_gp_2025 = get_event(2025, 'Australian Grand Prix')

aus_gp_2025

RoundNumber                                                          1
Country                                                      Australia
Location                                                     Melbourne
OfficialEventName    FORMULA 1 LOUIS VUITTON AUSTRALIAN GRAND PRIX ...
EventDate                                          2025-03-16 00:00:00
EventName                                        Australian Grand Prix
EventFormat                                               conventional
Session1                                                    Practice 1
Session1Date                                 2025-03-14 12:30:00+11:00
Session1DateUtc                                    2025-03-14 01:30:00
Session2                                                    Practice 2
Session2Date                                 2025-03-14 16:00:00+11:00
Session2DateUtc                                    2025-03-14 05:00:00
Session3                                                    Practice 3
Sessio

In [32]:
gp = get_event(2025, 1)
gpr = fastf1.get_session(2025, 1, 'R')
gpr.load()
results = gpr.results
results['GP name'] = gp['OfficialEventName']
results['country'] = gp['Country']
results['location'] = gp['Location']
results['driver'] = results['FullName'].apply(lambda x: x.split()[-1])
results = results[['driver', 'TeamName', 'GridPosition', 'Position','Status']]
results.reset_index(drop=True)
results

,driver,TeamName,GridPosition,Position,Status
4,Norris,McLaren,1.0,1.0,Finished
1,Verstappen,Red Bull Racing,3.0,2.0,Finished
63,Russell,Mercedes,4.0,3.0,Finished
12,Antonelli,Mercedes,16.0,4.0,Finished
23,Albon,Williams,6.0,5.0,Finished
18,Stroll,Aston Martin,13.0,6.0,Finished
27,Hulkenberg,Kick Sauber,17.0,7.0,Finished
16,Leclerc,Ferrari,7.0,8.0,Finished
81,Piastri,McLaren,2.0,9.0,Finished
44,Hamilton,Ferrari,8.0,10.0,Finished


In [4]:
results.dtypes

DriverId         object
TeamName         object
GridPosition    float64
Position        float64
Status           object
dtype: object

In [37]:
season2025 = []

for race in range(1, 12):
    
    gp = fastf1.get_event(2025, race)
    
    gpr = fastf1.get_session(2025, race, 'R')
    gpr.load()

    results = gpr.results.copy()

    results['GP name']  = gp['OfficialEventName']
    results['country']  = gp['Country']
    results['location'] = gp['Location']
    results['date'] = gp['EventDate']
    results['driver'] = results['FullName'].apply(lambda x: x.split()[-1])

    keep = ['GP name', 'date', 'driver', 'TeamName', 'GridPosition', 'Position', 'Status', 'country', 'location']
    results = results[keep].reset_index(drop=True)

    season2025.append(results)

season2025 = pd.concat(season2025, ignore_index=True)
season2025

,GP name,date,driver,TeamName,GridPosition,Position,Status,country,location
0,FORMULA 1 LOUIS VUITTON AUSTRALIAN GRAND PRIX ...,2025-03-16,Norris,McLaren,1.0,1.0,Finished,Australia,Melbourne
1,FORMULA 1 LOUIS VUITTON AUSTRALIAN GRAND PRIX ...,2025-03-16,Verstappen,Red Bull Racing,3.0,2.0,Finished,Australia,Melbourne
2,FORMULA 1 LOUIS VUITTON AUSTRALIAN GRAND PRIX ...,2025-03-16,Russell,Mercedes,4.0,3.0,Finished,Australia,Melbourne
3,FORMULA 1 LOUIS VUITTON AUSTRALIAN GRAND PRIX ...,2025-03-16,Antonelli,Mercedes,16.0,4.0,Finished,Australia,Melbourne
4,FORMULA 1 LOUIS VUITTON AUSTRALIAN GRAND PRIX ...,2025-03-16,Albon,Williams,6.0,5.0,Finished,Australia,Melbourne
...,...,...,...,...,...,...,...,...,...
214,FORMULA 1 MSC CRUISES AUSTRIAN GRAND PRIX 2025,2025-06-29,Tsunoda,Red Bull Racing,18.0,16.0,Lapped,Austria,Spielberg
215,FORMULA 1 MSC CRUISES AUSTRIAN GRAND PRIX 2025,2025-06-29,Albon,Williams,12.0,17.0,Retired,Austria,Spielberg
216,FORMULA 1 MSC CRUISES AUSTRIAN GRAND PRIX 2025,2025-06-29,Verstappen,Red Bull Racing,7.0,18.0,Retired,Austria,Spielberg
217,FORMULA 1 MSC CRUISES AUSTRIAN GRAND PRIX 2025,2025-06-29,Antonelli,Mercedes,9.0,19.0,Retired,Austria,Spielberg


In [38]:
drivers = pd.read_csv('./f1_datasets/drivers.csv')
circuits = pd.read_csv('./f1_datasets/circuits.csv')
constructors = pd.read_csv('./f1_datasets/constructors.csv')
season2025 = season2025.rename(columns={
    'GridPosition': 'start',
    'Position': 'finish',
    'TeamName':'team',
})
constructors = constructors.rename(columns={
    'name': 'team',
})

In [39]:
season2025['start'] = season2025['start'].astype('Int64')
season2025['finish'] = season2025['finish'].astype('Int64')

In [40]:
gp_map = {
    r'.*AUSTRALIAN.*': 'Australian Grand Prix',
    r'.*CHINESE.*': 'Chinese Grand Prix',
    r'.*JAPANESE.*': 'Japanese Grand Prix',
    r'.*BAHRAIN.*': 'Bahrain Grand Prix',
    r'.*SAUDI.*': 'Saudi Arabian Grand Prix',
    r'.*MIAMI.*': 'Miami Grand Prix',
    r'.*EMILIA.*': 'Emilia-Romagna Grand Prix',
    r'.*MONACO.*': 'Monaco Grand Prix',
    r'.*SPAIN.*': 'Spanish Grand Prix',
    r'.*CANADA.*': 'Canadian Grand Prix',
    r'.*AUSTRIA.*': 'Austrian Grand Prix'
}
team_map = {
    'Red Bull Racing': 'Red Bull',
    'Kick Sauber': 'Sauber',
    'Racing Bulls': 'RB F1 Team'
}
season2025['team'] = season2025['team'].replace(team_map)
season2025['GP name'] = season2025['GP name'].replace(gp_map, regex=True)

In [42]:
drivers = drivers.rename(columns={'surname':'driver'})

In [43]:
season2025 = season2025.merge(
    drivers[['driver', 'dob', 'nationality']],
    on='driver',
    how='left'
)
season2025 = season2025.merge(
    circuits[['location', 'name']],
    on='location',
    how='left'
)
season2025 = season2025.merge(
    constructors[['team','nationality']],
    on='team',
    how='left'
)
season2025

,GP name,date,driver,team,start,finish,Status,country,location,dob,nationality_x,name,nationality_y
0,Australian Grand Prix,2025-03-16,Norris,McLaren,1,1,Finished,Australia,Melbourne,1999-11-13,British,Albert Park Grand Prix Circuit,British
1,Australian Grand Prix,2025-03-16,Verstappen,Red Bull,3,2,Finished,Australia,Melbourne,1972-03-04,Dutch,Albert Park Grand Prix Circuit,Austrian
2,Australian Grand Prix,2025-03-16,Verstappen,Red Bull,3,2,Finished,Australia,Melbourne,1997-09-30,Dutch,Albert Park Grand Prix Circuit,Austrian
3,Australian Grand Prix,2025-03-16,Russell,Mercedes,4,3,Finished,Australia,Melbourne,1998-02-15,British,Albert Park Grand Prix Circuit,German
4,Australian Grand Prix,2025-03-16,Antonelli,Mercedes,16,4,Finished,Australia,Melbourne,NaN,NaN,Albert Park Grand Prix Circuit,German
...,...,...,...,...,...,...,...,...,...,...,...,...,...
257,Austrian Grand Prix,2025-06-29,Albon,Williams,12,17,Retired,Austria,Spielberg,1996-03-23,Thai,Red Bull Ring,British
258,Austrian Grand Prix,2025-06-29,Verstappen,Red Bull,7,18,Retired,Austria,Spielberg,1972-03-04,Dutch,Red Bull Ring,Austrian
259,Austrian Grand Prix,2025-06-29,Verstappen,Red Bull,7,18,Retired,Austria,Spielberg,1997-09-30,Dutch,Red Bull Ring,Austrian
260,Austrian Grand Prix,2025-06-29,Antonelli,Mercedes,9,19,Retired,Austria,Spielberg,NaN,NaN,Red Bull Ring,German


In [44]:
season2025.dtypes

GP name                  object
date             datetime64[ns]
driver                   object
team                     object
start                     Int64
finish                    Int64
Status                   object
country                  object
location                 object
dob                      object
nationality_x            object
name                       None
nationality_y            object
dtype: object

In [45]:
season2025['Status'].unique()

array(['Finished', 'Retired', 'Lapped', 'Disqualified', 'Did not start'],
      dtype=object)

In [47]:
season2025['dob'] = pd.to_datetime(season2025['dob'], errors='coerce')
season2025['statusId'] = season2025['Status'].apply(
    lambda s: 1 if s in {'Finished', 'Lapped', 'Disqualified'} else 3
)
season2025['year'] = 2025

season2025 = season2025.reset_index(drop=True)
season2025 = season2025.rename(columns={
    'name': 'circuit name',
    'nationality_x':'driver_home',
    'nationality_y':'team_home'
})

final_cols = [
    'year', 'GP name', 'date', 'start', 'finish', 'statusId',
    'team', 'driver', 'dob', 'circuit name', 'location', 'country', 'driver_home', 'team_home'
]
season2025 = season2025[final_cols]

season2025

,year,GP name,date,start,finish,statusId,team,driver,dob,circuit name,location,country,driver_home,team_home
0,2025,Australian Grand Prix,2025-03-16,1,1,1,McLaren,Norris,1999-11-13,Albert Park Grand Prix Circuit,Melbourne,Australia,British,British
1,2025,Australian Grand Prix,2025-03-16,3,2,1,Red Bull,Verstappen,1972-03-04,Albert Park Grand Prix Circuit,Melbourne,Australia,Dutch,Austrian
2,2025,Australian Grand Prix,2025-03-16,3,2,1,Red Bull,Verstappen,1997-09-30,Albert Park Grand Prix Circuit,Melbourne,Australia,Dutch,Austrian
3,2025,Australian Grand Prix,2025-03-16,4,3,1,Mercedes,Russell,1998-02-15,Albert Park Grand Prix Circuit,Melbourne,Australia,British,German
4,2025,Australian Grand Prix,2025-03-16,16,4,1,Mercedes,Antonelli,NaT,Albert Park Grand Prix Circuit,Melbourne,Australia,NaN,German
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
257,2025,Austrian Grand Prix,2025-06-29,12,17,3,Williams,Albon,1996-03-23,Red Bull Ring,Spielberg,Austria,Thai,British
258,2025,Austrian Grand Prix,2025-06-29,7,18,3,Red Bull,Verstappen,1972-03-04,Red Bull Ring,Spielberg,Austria,Dutch,Austrian
259,2025,Austrian Grand Prix,2025-06-29,7,18,3,Red Bull,Verstappen,1997-09-30,Red Bull Ring,Spielberg,Austria,Dutch,Austrian
260,2025,Austrian Grand Prix,2025-06-29,9,19,3,Mercedes,Antonelli,NaT,Red Bull Ring,Spielberg,Austria,NaN,German


In [48]:
def nationality(x):
    x = str(x).strip().lower()
    if x in ['austrian', 'austria']:
        return 'AUT'
    if x in ['australian', 'australia']:
        return 'AUS'
    if x in ['indian', 'india']:
        return 'IND'
    if x in ['indonesian', 'indonesia']:
        return 'INA'
    if x in ['uk', 'british', 'england', 'great britain']:
        return 'BRI'
    if x in ['usa', 'united states', 'american']:
        return 'AME'
    if x in ['fra', 'france', 'french']:
        return 'FRE'
    return x[:3].upper()

season2025['driver_home'] = season2025['driver_home'].apply(nationality)
season2025['team_home'] = season2025['team_home'].apply(nationality)
season2025['country'] = season2025['country'].apply(nationality)

season2025['driver home'] = (season2025['driver_home'] == season2025['country']).astype(int)
season2025['team home'] = (season2025['team_home'] == season2025['country']).astype(int)

In [49]:
season2025 = season2025.drop(columns=[ 'driver_home', 'team_home'])
season2025 = season2025.reset_index(drop=True)
season2025

,year,GP name,date,start,finish,statusId,team,driver,dob,circuit name,location,country,driver home,team home
0,2025,Australian Grand Prix,2025-03-16,1,1,1,McLaren,Norris,1999-11-13,Albert Park Grand Prix Circuit,Melbourne,AUS,0,0
1,2025,Australian Grand Prix,2025-03-16,3,2,1,Red Bull,Verstappen,1972-03-04,Albert Park Grand Prix Circuit,Melbourne,AUS,0,0
2,2025,Australian Grand Prix,2025-03-16,3,2,1,Red Bull,Verstappen,1997-09-30,Albert Park Grand Prix Circuit,Melbourne,AUS,0,0
3,2025,Australian Grand Prix,2025-03-16,4,3,1,Mercedes,Russell,1998-02-15,Albert Park Grand Prix Circuit,Melbourne,AUS,0,0
4,2025,Australian Grand Prix,2025-03-16,16,4,1,Mercedes,Antonelli,NaT,Albert Park Grand Prix Circuit,Melbourne,AUS,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
257,2025,Austrian Grand Prix,2025-06-29,12,17,3,Williams,Albon,1996-03-23,Red Bull Ring,Spielberg,AUT,0,0
258,2025,Austrian Grand Prix,2025-06-29,7,18,3,Red Bull,Verstappen,1972-03-04,Red Bull Ring,Spielberg,AUT,0,1
259,2025,Austrian Grand Prix,2025-06-29,7,18,3,Red Bull,Verstappen,1997-09-30,Red Bull Ring,Spielberg,AUT,0,1
260,2025,Austrian Grand Prix,2025-06-29,9,19,3,Mercedes,Antonelli,NaT,Red Bull Ring,Spielberg,AUT,0,0


In [50]:
season2025.to_csv('season2025.csv')

In [17]:
session = fastf1.get_session(2018,3,'R')
session.load()
gprw = session.weather_data
gprw

,Time,AirTemp,Humidity,Pressure,Rainfall,TrackTemp,WindDirection,WindSpeed
0,0 days 00:00:32.761000,19.0,21.2,1018.3,False,39.0,104,1.8
1,0 days 00:01:32.775000,18.9,21.5,1018.3,False,39.0,86,2.2
2,0 days 00:02:32.789000,19.0,21.9,1018.4,False,39.0,117,2.2
3,0 days 00:03:32.804000,19.1,22.3,1018.3,False,39.0,107,1.9
4,0 days 00:04:32.821000,19.2,22.4,1018.3,False,39.0,104,1.4
...,...,...,...,...,...,...,...,...
107,0 days 01:47:34.214000,19.6,25.6,1018.1,False,36.5,120,1.1
108,0 days 01:48:34.228000,19.4,25.8,1018.1,False,36.5,118,0.9
109,0 days 01:49:34.240000,19.2,27.0,1018.1,False,36.5,152,1.2
110,0 days 01:50:34.255000,19.1,26.8,1018.1,True,28.3,84,0.8


In [2]:
def weather_2018():
    sched = fastf1.get_event_schedule(2018)
    sched = sched[sched['EventFormat'].str.lower() != 'testing']

    def _one(evt):
        try:
            s = fastf1.get_session(2018, evt['RoundNumber'], 'R')
            s.load(laps=False, telemetry=False, weather=True, messages=False)
            w = s.weather_data
            if w.empty:
                return None
            return {
                'date': str(evt['EventDate'])[:10],
                'avg_air_temp': round(w['AirTemp'].mean(), 1),
                'max_air_temp': w['AirTemp'].max(),
                'min_air_temp': w['AirTemp'].min(),
                'rainfall': int(w['Rainfall'].any()),
                'avg_humidity': round(w['Humidity'].mean(), 1),
                'max_humidity': w['Humidity'].max(),
                'min_humidity': w['Humidity'].min(),
                'avg_track_temp': round(w['TrackTemp'].mean(), 1),
                'max_track_temp': w['TrackTemp'].max(),
                'min_track_temp': w['TrackTemp'].min(),
            }
        except Exception as e:
            print(f"Skipped {evt['EventName']}: {e}")
            return None

    with ThreadPoolExecutor(max_workers=8) as pool:
        rows = list(filter(None, pool.map(_one, sched.to_dict('records'))))
    return pd.DataFrame(rows)

weather_df_2018 = weather_2018()
weather_df_2018

,date,avg_air_temp,max_air_temp,min_air_temp,rainfall,avg_humidity,max_humidity,min_humidity,avg_track_temp,max_track_temp,min_track_temp
0,2018-03-25,24.1,24.8,23.3,1,30.9,37.2,25.6,36.3,38.9,32.4
1,2018-04-08,28.0,28.3,27.5,0,47.4,52.9,44.5,32.2,33.5,31.1
2,2018-04-15,19.4,20.2,18.8,1,24.1,27.0,21.2,37.0,39.0,28.3
3,2018-04-29,16.7,17.2,16.3,0,45.7,47.8,42.2,25.3,28.0,23.4
4,2018-05-13,16.1,17.1,14.9,1,52.3,55.7,49.1,32.3,36.7,28.8
5,2018-05-27,26.6,27.6,25.3,1,50.9,59.3,43.3,34.6,37.0,33.2
6,2018-06-10,20.5,21.3,19.7,0,27.8,29.9,25.7,36.9,42.1,32.4
7,2018-06-24,24.5,25.6,23.4,1,49.0,55.9,44.6,42.0,45.2,37.6
8,2018-07-01,23.2,24.2,22.2,0,37.1,39.8,34.3,45.9,48.6,43.3
9,2018-07-08,27.4,28.6,26.4,0,38.7,40.7,36.1,52.2,54.1,47.9


In [3]:
def weather_2019():
    sched = fastf1.get_event_schedule(2019)
    sched = sched[sched['EventFormat'].str.lower() != 'testing']

    def _one(evt):
        try:
            s = fastf1.get_session(2019, evt['RoundNumber'], 'R')
            s.load(laps=False, telemetry=False, weather=True, messages=False)
            w = s.weather_data
            if w.empty:
                return None
            return {
                'date': str(evt['EventDate'])[:10],
                'avg_air_temp': round(w['AirTemp'].mean(), 1),
                'max_air_temp': w['AirTemp'].max(),
                'min_air_temp': w['AirTemp'].min(),
                'rainfall': int(w['Rainfall'].any()),
                'avg_humidity': round(w['Humidity'].mean(), 1),
                'max_humidity': w['Humidity'].max(),
                'min_humidity': w['Humidity'].min(),
                'avg_track_temp': round(w['TrackTemp'].mean(), 1),
                'max_track_temp': w['TrackTemp'].max(),
                'min_track_temp': w['TrackTemp'].min(),
            }
        except Exception as e:
            print(f"Skipped {evt['EventName']}: {e}")
            return None

    with ThreadPoolExecutor(max_workers=8) as pool:
        rows = list(filter(None, pool.map(_one, sched.to_dict('records'))))
    return pd.DataFrame(rows)

weather_df_2019 = weather_2019()
weather_df_2019

,date,avg_air_temp,max_air_temp,min_air_temp,rainfall,avg_humidity,max_humidity,min_humidity,avg_track_temp,max_track_temp,min_track_temp
0,2019-03-17,23.5,24.1,22.8,0,70.5,72.4,68.3,41.3,45.0,37.1
1,2019-03-31,26.2,26.6,25.9,0,53.4,56.3,48.8,28.6,29.5,27.4
2,2019-04-14,19.3,19.8,18.9,0,46.0,51.1,42.8,27.3,30.6,25.1
3,2019-04-28,19.8,21.0,18.5,0,51.4,57.5,45.8,39.7,43.6,34.2
4,2019-05-12,20.1,21.2,19.5,0,54.7,60.5,47.6,40.7,42.4,38.7
5,2019-05-26,22.8,24.0,21.9,1,52.7,54.8,48.9,32.7,35.3,30.4
6,2019-06-09,29.4,30.9,28.1,0,16.1,19.8,12.9,51.1,52.6,49.5
7,2019-06-23,26.9,27.9,25.3,0,39.1,43.0,36.3,54.5,57.2,50.6
8,2019-06-30,34.4,35.3,33.2,0,18.0,21.5,15.5,50.7,51.9,49.0
9,2019-07-14,18.9,20.2,17.6,0,62.1,69.1,57.2,30.3,35.6,27.5


In [4]:
def weather_2020():
    sched = fastf1.get_event_schedule(2020)
    sched = sched[sched['EventFormat'].str.lower() != 'testing']

    def _one(evt):
        try:
            s = fastf1.get_session(2020, evt['RoundNumber'], 'R')
            s.load(laps=False, telemetry=False, weather=True, messages=False)
            w = s.weather_data
            if w.empty:
                return None
            return {
                'date': str(evt['EventDate'])[:10],
                'avg_air_temp': round(w['AirTemp'].mean(), 1),
                'max_air_temp': w['AirTemp'].max(),
                'min_air_temp': w['AirTemp'].min(),
                'rainfall': int(w['Rainfall'].any()),
                'avg_humidity': round(w['Humidity'].mean(), 1),
                'max_humidity': w['Humidity'].max(),
                'min_humidity': w['Humidity'].min(),
                'avg_track_temp': round(w['TrackTemp'].mean(), 1),
                'max_track_temp': w['TrackTemp'].max(),
                'min_track_temp': w['TrackTemp'].min(),
            }
        except Exception as e:
            print(f"Skipped {evt['EventName']}: {e}")
            return None

    with ThreadPoolExecutor(max_workers=8) as pool:
        rows = list(filter(None, pool.map(_one, sched.to_dict('records'))))
    return pd.DataFrame(rows)

weather_df_2020 = weather_2020()
weather_df_2020

,date,avg_air_temp,max_air_temp,min_air_temp,rainfall,avg_humidity,max_humidity,min_humidity,avg_track_temp,max_track_temp,min_track_temp
0,2020-07-05,28.7,30.0,26.9,0,33.8,36.5,32.0,52.0,55.6,46.7
1,2020-07-12,20.8,22.1,19.4,1,34.0,37.6,31.0,39.9,43.7,35.6
2,2020-07-19,20.7,21.8,19.9,1,77.2,81.0,72.5,27.1,30.2,23.9
3,2020-08-02,21.3,22.1,20.8,0,44.2,46.2,41.4,40.6,43.3,36.9
4,2020-08-09,25.3,26.8,23.5,0,58.6,63.8,54.3,43.2,45.1,41.0
5,2020-08-16,29.8,30.7,28.6,0,56.4,60.2,53.6,47.6,50.3,42.8
6,2020-08-30,18.6,20.1,17.4,0,54.9,60.5,50.4,29.8,32.2,26.9
7,2020-09-06,28.1,28.9,27.3,0,40.1,42.3,38.0,41.2,45.5,35.6
8,2020-09-13,30.2,30.7,29.6,0,39.9,42.6,37.6,43.1,45.8,38.8
9,2020-09-27,28.8,30.5,27.6,0,51.4,56.5,45.0,40.3,42.0,37.6


In [5]:
def weather_2021():
    sched = fastf1.get_event_schedule(2021)
    sched = sched[sched['EventFormat'].str.lower() != 'testing']

    def _one(evt):
        try:
            s = fastf1.get_session(2021, evt['RoundNumber'], 'R')
            s.load(laps=False, telemetry=False, weather=True, messages=False)
            w = s.weather_data
            if w.empty:
                return None
            return {
                'date': str(evt['EventDate'])[:10],
                'avg_air_temp': round(w['AirTemp'].mean(), 1),
                'max_air_temp': w['AirTemp'].max(),
                'min_air_temp': w['AirTemp'].min(),
                'rainfall': int(w['Rainfall'].any()),
                'avg_humidity': round(w['Humidity'].mean(), 1),
                'max_humidity': w['Humidity'].max(),
                'min_humidity': w['Humidity'].min(),
                'avg_track_temp': round(w['TrackTemp'].mean(), 1),
                'max_track_temp': w['TrackTemp'].max(),
                'min_track_temp': w['TrackTemp'].min(),
            }
        except Exception as e:
            print(f"Skipped {evt['EventName']}: {e}")
            return None

    with ThreadPoolExecutor(max_workers=8) as pool:
        rows = list(filter(None, pool.map(_one, sched.to_dict('records'))))
    return pd.DataFrame(rows)

weather_df_2021 = weather_2021()
weather_df_2021

,date,avg_air_temp,max_air_temp,min_air_temp,rainfall,avg_humidity,max_humidity,min_humidity,avg_track_temp,max_track_temp,min_track_temp
0,2021-03-28,20.5,20.9,20.4,0,54.5,56.8,52.6,27.6,30.2,26.0
1,2021-04-18,10.6,12.3,9.2,1,69.5,77.1,61.6,16.8,18.6,13.8
2,2021-05-02,19.4,20.0,18.1,0,39.4,45.5,36.7,39.0,40.6,33.8
3,2021-05-09,22.2,23.5,21.3,0,57.6,61.1,54.2,32.9,36.2,30.7
4,2021-05-23,20.8,21.5,20.3,0,61.4,66.1,57.2,37.6,43.4,33.0
5,2021-06-06,24.2,25.5,22.9,0,55.8,61.8,48.0,38.3,44.3,32.8
6,2021-06-20,25.3,27.4,23.8,0,58.6,68.0,49.2,35.2,38.9,30.1
7,2021-06-27,26.7,27.3,26.0,0,33.2,42.3,30.0,52.4,54.4,48.1
8,2021-07-04,21.0,22.6,19.0,1,49.7,58.8,45.6,33.2,36.9,27.6
9,2021-07-18,29.4,30.1,28.4,0,37.2,41.3,33.4,50.9,53.1,48.3


In [6]:
def weather_2022():
    sched = fastf1.get_event_schedule(2022)
    sched = sched[sched['EventFormat'].str.lower() != 'testing']

    def _one(evt):
        try:
            s = fastf1.get_session(2022, evt['RoundNumber'], 'R')
            s.load(laps=False, telemetry=False, weather=True, messages=False)
            w = s.weather_data
            if w.empty:
                return None
            return {
                'date': str(evt['EventDate'])[:10],
                'avg_air_temp': round(w['AirTemp'].mean(), 1),
                'max_air_temp': w['AirTemp'].max(),
                'min_air_temp': w['AirTemp'].min(),
                'rainfall': int(w['Rainfall'].any()),
                'avg_humidity': round(w['Humidity'].mean(), 1),
                'max_humidity': w['Humidity'].max(),
                'min_humidity': w['Humidity'].min(),
                'avg_track_temp': round(w['TrackTemp'].mean(), 1),
                'max_track_temp': w['TrackTemp'].max(),
                'min_track_temp': w['TrackTemp'].min(),
            }
        except Exception as e:
            print(f"Skipped {evt['EventName']}: {e}")
            return None

    with ThreadPoolExecutor(max_workers=8) as pool:
        rows = list(filter(None, pool.map(_one, sched.to_dict('records'))))
    return pd.DataFrame(rows)

weather_df_2022 = weather_2022()
weather_df_2022

,date,avg_air_temp,max_air_temp,min_air_temp,rainfall,avg_humidity,max_humidity,min_humidity,avg_track_temp,max_track_temp,min_track_temp
0,2022-03-20,23.6,25.7,22.3,0,29.5,38.0,16.0,28.6,32.3,26.3
1,2022-03-27,25.4,25.6,25.3,0,59.7,65.0,56.0,28.4,29.8,27.2
2,2022-04-10,26.7,27.4,25.8,0,43.7,48.0,40.0,37.4,41.2,31.9
3,2022-04-24,12.9,14.2,11.1,0,83.2,92.0,75.0,17.4,19.1,15.6
4,2022-05-08,30.6,32.3,29.4,1,58.9,67.0,47.0,42.8,48.3,36.2
5,2022-05-22,36.6,37.2,35.9,0,7.1,11.0,5.0,49.0,49.7,47.6
6,2022-05-29,22.3,24.3,20.9,1,75.7,88.0,64.0,31.0,42.1,26.3
7,2022-06-12,25.4,26.1,24.9,0,59.5,65.0,52.0,47.1,50.9,40.9
8,2022-06-19,20.8,22.3,18.8,0,28.9,35.0,24.0,41.3,42.5,38.9
9,2022-07-03,18.3,19.2,17.5,1,59.7,65.0,53.0,30.0,34.1,27.3


In [20]:
def weather_2023():
    sched = fastf1.get_event_schedule(2023)
    sched = sched[sched['EventFormat'].str.lower() != 'testing']

    def _one(evt):
        try:
            s = fastf1.get_session(2023, evt['RoundNumber'], 'R')
            s.load(laps=False, telemetry=False, weather=True, messages=False)
            w = s.weather_data
            if w.empty:
                return None
            return {
                'date': str(evt['EventDate'])[:10],
                'avg_air_temp': round(w['AirTemp'].mean(), 1),
                'max_air_temp': w['AirTemp'].max(),
                'min_air_temp': w['AirTemp'].min(),
                'rainfall': int(w['Rainfall'].any()),
                'avg_humidity': round(w['Humidity'].mean(), 1),
                'max_humidity': w['Humidity'].max(),
                'min_humidity': w['Humidity'].min(),
                'avg_track_temp': round(w['TrackTemp'].mean(), 1),
                'max_track_temp': w['TrackTemp'].max(),
                'min_track_temp': w['TrackTemp'].min(),
            }
        except Exception as e:
            print(f"Skipped {evt['EventName']}: {e}")
            return None

    with ThreadPoolExecutor(max_workers=8) as pool:
        rows = list(filter(None, pool.map(_one, sched.to_dict('records'))))
    return pd.DataFrame(rows)

weather_df_2023 = weather_2023()
weather_df_2023

Unable to deserialize response: a bytes-like object is required, not 'NoneType'


,date,avg_air_temp,max_air_temp,min_air_temp,rainfall,avg_humidity,max_humidity,min_humidity,avg_track_temp,max_track_temp,min_track_temp
0,2023-03-05,27.4,29.8,26.2,0,21.5,28.0,18.0,31.0,35.1,28.7
1,2023-03-19,26.1,26.4,25.8,0,57.8,62.0,55.0,31.8,34.3,29.9
2,2023-04-02,17.4,18.1,16.8,0,54.2,61.0,50.0,30.1,35.4,19.2
3,2023-04-30,24.9,25.6,23.9,0,49.2,53.0,46.0,41.2,44.8,34.5
4,2023-05-07,27.1,27.8,26.6,0,59.4,62.0,55.0,36.7,52.5,32.8
5,2023-05-28,25.1,26.3,23.0,1,45.9,67.0,39.0,39.3,48.3,27.2
6,2023-06-04,22.7,24.7,21.5,0,63.1,69.0,54.0,33.8,42.5,26.8
7,2023-06-18,18.5,19.5,17.0,0,63.9,69.0,60.0,30.6,34.3,27.0
8,2023-07-02,22.4,23.2,21.5,1,52.0,56.0,48.0,32.1,34.5,30.5
9,2023-07-09,22.4,23.2,21.5,1,52.0,56.0,48.0,32.1,34.5,30.5


In [8]:
def weather_2024():
    sched = fastf1.get_event_schedule(2024)
    sched = sched[sched['EventFormat'].str.lower() != 'testing']

    def _one(evt):
        try:
            s = fastf1.get_session(2024, evt['RoundNumber'], 'R')
            s.load(laps=False, telemetry=False, weather=True, messages=False)
            w = s.weather_data
            if w.empty:
                return None
            return {
                'date': str(evt['EventDate'])[:10],
                'avg_air_temp': round(w['AirTemp'].mean(), 1),
                'max_air_temp': w['AirTemp'].max(),
                'min_air_temp': w['AirTemp'].min(),
                'rainfall': int(w['Rainfall'].any()),
                'avg_humidity': round(w['Humidity'].mean(), 1),
                'max_humidity': w['Humidity'].max(),
                'min_humidity': w['Humidity'].min(),
                'avg_track_temp': round(w['TrackTemp'].mean(), 1),
                'max_track_temp': w['TrackTemp'].max(),
                'min_track_temp': w['TrackTemp'].min(),
            }
        except Exception as e:
            print(f"Skipped {evt['EventName']}: {e}")
            return None

    with ThreadPoolExecutor(max_workers=8) as pool:
        rows = list(filter(None, pool.map(_one, sched.to_dict('records'))))
    return pd.DataFrame(rows)

weather_df_2024 = weather_2024()
weather_df_2024

,date,avg_air_temp,max_air_temp,min_air_temp,rainfall,avg_humidity,max_humidity,min_humidity,avg_track_temp,max_track_temp,min_track_temp
0,2024-03-02,18.2,18.9,17.6,0,48.8,51.0,45.0,23.7,26.5,21.9
1,2024-03-09,25.5,25.9,25.2,0,62.3,67.0,57.0,31.6,33.6,29.1
2,2024-03-24,20.6,22.0,19.0,0,44.5,49.0,42.0,38.4,39.6,36.7
3,2024-04-07,21.7,22.7,19.5,0,43.4,57.0,32.0,37.5,40.5,31.3
4,2024-04-21,18.6,19.1,18.0,0,66.1,69.0,63.0,29.8,32.3,26.8
5,2024-05-05,28.5,29.0,27.9,0,59.0,63.0,56.0,44.7,49.1,39.3
6,2024-05-19,25.1,26.1,24.0,0,48.4,55.0,42.0,42.0,46.0,38.6
7,2024-05-26,21.6,22.3,20.9,0,64.0,69.0,57.0,46.3,50.0,40.2
8,2024-06-09,17.8,19.1,15.9,1,73.4,80.0,68.0,24.6,28.5,19.9
9,2024-06-23,24.1,24.7,23.6,1,63.4,68.0,60.0,41.1,43.6,38.3


In [9]:
def weather_2025():
    sched = fastf1.get_event_schedule(2025)
    sched = sched[sched['EventFormat'].str.lower() != 'testing']

    def _one(evt):
        try:
            s = fastf1.get_session(2025, evt['RoundNumber'], 'R')
            s.load(laps=False, telemetry=False, weather=True, messages=False)
            w = s.weather_data
            if w.empty:
                return None
            return {
                'date': str(evt['EventDate'])[:10],
                'avg_air_temp': round(w['AirTemp'].mean(), 1),
                'max_air_temp': w['AirTemp'].max(),
                'min_air_temp': w['AirTemp'].min(),
                'rainfall': int(w['Rainfall'].any()),
                'avg_humidity': round(w['Humidity'].mean(), 1),
                'max_humidity': w['Humidity'].max(),
                'min_humidity': w['Humidity'].min(),
                'avg_track_temp': round(w['TrackTemp'].mean(), 1),
                'max_track_temp': w['TrackTemp'].max(),
                'min_track_temp': w['TrackTemp'].min(),
            }
        except Exception as e:
            print(f"Skipped {evt['EventName']}: {e}")
            return None

    with ThreadPoolExecutor(max_workers=8) as pool:
        rows = list(filter(None, pool.map(_one, sched.to_dict('records'))))
    return pd.DataFrame(rows)

weather_df_2025 = weather_2025()
weather_df_2025

Skipped Mexico City Grand Prix: The data you are trying to access has not been loaded yet. See `Session.load`
Skipped Hungarian Grand Prix: The data you are trying to access has not been loaded yet. See `Session.load`
Skipped Azerbaijan Grand Prix: The data you are trying to access has not been loaded yet. See `Session.load`
Skipped Italian Grand Prix: The data you are trying to access has not been loaded yet. See `Session.load`
Skipped Singapore Grand Prix: The data you are trying to access has not been loaded yet. See `Session.load`
Skipped Dutch Grand Prix: The data you are trying to access has not been loaded yet. See `Session.load`
Skipped São Paulo Grand Prix: The data you are trying to access has not been loaded yet. See `Session.load`
Skipped United States Grand Prix: The data you are trying to access has not been loaded yet. See `Session.load`
Skipped Abu Dhabi Grand Prix: The data you are trying to access has not been loaded yet. See `Session.load`
Skipped Las Vegas Grand Pri

,date,avg_air_temp,max_air_temp,min_air_temp,rainfall,avg_humidity,max_humidity,min_humidity,avg_track_temp,max_track_temp,min_track_temp
0,2025-03-16,15.7,16.6,15.1,1,78.4,92.0,68.0,18.9,19.4,18.3
1,2025-03-23,27.2,28.1,26.4,0,18.2,22.0,14.0,35.8,43.0,30.5
2,2025-04-06,14.4,14.7,14.2,0,76.4,78.0,73.0,20.5,23.0,19.0
3,2025-04-13,27.1,27.9,26.6,0,45.3,48.0,41.0,31.9,34.8,30.2
4,2025-04-20,29.5,31.7,28.3,0,62.7,74.0,47.0,37.8,39.4,35.9
5,2025-05-04,26.6,28.0,25.9,1,63.0,69.0,59.0,38.7,45.2,34.8
6,2025-05-18,24.0,24.3,23.4,0,38.3,40.0,36.0,42.7,45.9,38.6
7,2025-05-25,22.2,22.9,21.3,0,52.5,56.0,48.0,43.3,45.2,41.3
8,2025-06-01,29.1,30.0,28.1,0,54.6,62.0,48.0,48.5,50.4,46.5
9,2025-06-15,24.0,25.6,22.8,1,28.0,34.0,25.0,49.3,50.4,47.0


In [10]:
weather_df_2025 = weather_df_2025.iloc[:-2]

In [11]:
weather = pd.concat([weather_df_2018,weather_df_2019,weather_df_2020,weather_df_2021,weather_df_2022,weather_df_2023,weather_df_2024,weather_df_2025], ignore_index=True)
weather

,date,avg_air_temp,max_air_temp,min_air_temp,rainfall,avg_humidity,max_humidity,min_humidity,avg_track_temp,max_track_temp,min_track_temp
0,2018-03-25,24.1,24.8,23.3,1,30.9,37.2,25.6,36.3,38.9,32.4
1,2018-04-08,28.0,28.3,27.5,0,47.4,52.9,44.5,32.2,33.5,31.1
2,2018-04-15,19.4,20.2,18.8,1,24.1,27.0,21.2,37.0,39.0,28.3
3,2018-04-29,16.7,17.2,16.3,0,45.7,47.8,42.2,25.3,28.0,23.4
4,2018-05-13,16.1,17.1,14.9,1,52.3,55.7,49.1,32.3,36.7,28.8
...,...,...,...,...,...,...,...,...,...,...,...
155,2025-05-18,24.0,24.3,23.4,0,38.3,40.0,36.0,42.7,45.9,38.6
156,2025-05-25,22.2,22.9,21.3,0,52.5,56.0,48.0,43.3,45.2,41.3
157,2025-06-01,29.1,30.0,28.1,0,54.6,62.0,48.0,48.5,50.4,46.5
158,2025-06-15,24.0,25.6,22.8,1,28.0,34.0,25.0,49.3,50.4,47.0


In [ ]:
weather['date'] = pd.to_datetime(weather['date'])


In [17]:
weather.dtypes

date              datetime64[ns]
avg_air_temp             float64
max_air_temp             float64
min_air_temp             float64
rainfall                   int64
avg_humidity             float64
max_humidity             float64
min_humidity             float64
avg_track_temp           float64
max_track_temp           float64
min_track_temp           float64
dtype: object

In [18]:
weather.to_csv('weather.csv')